# Graphs — User Guide

`StarLayerGraph` extends rdflib's `Graph` to support RDF 1.2: triple terms, reification, and direction-tagged literals. This guide also covers RDFC-1.0 canonical hashing/graph comparison, a general graph feature that isn't RDF-1.2-specific but lives in this same package.

Two closely related guides broke out of this one and are worth reading alongside it: **[2.a Working with datasets](02a-graphs-datasets.ipynb)** (`StarLayerDataset`, multiple named graphs in one store) and **[2.b Serialization formats](02b-graphs-serialization-formats.ipynb)** (all eight RDF 1.2 formats `parse()`/`serialize()` support).

Two more guides worth knowing about: for SHACL shape validation over the graphs built here, see the [SHACL shapes guide](04-shacl-shapes.ipynb). For a worked, end-to-end use of the canonical hashing covered in section 2 below (committing a hash of exactly the part of a graph a shape covers, then re-verifying it later), see the [subgraph extraction guide](04f-shacl-subgraph-extraction.ipynb)'s final section.

## How to run this notebook

1. `pip install "git+https://github.com/hidden-graph/starlayer.git"` (or install the three packages editable from a local checkout — see the root [README](../../README.md)).
2. Run cells from top to bottom — later sections reuse variables from earlier ones.

In [ ]:
from starlayergraph import StarLayerGraph, Namespace, TripleTerm, DirLangString, Literal, BNode
from starlayergraph.compare import isomorphic
from starlayergraph.rdfc import rdfc10_hash, to_canonical_nquads

EX = Namespace("http://example.org/")
RDF = Namespace("http://www.w3.org/1999/02/22-rdf-syntax-ns#")

## 1. Graph and literal semantics

Extension of the rdflib graph model to support RDF 1.2:
- Triple terms and statement resources
- Reification via `rdf:reifies` and statement metadata
- Direction-tagged strings such as `"hello"@en--ltr` and `"مرحبا"@ar--rtl`

In [2]:
# create the graph, and assign a namespace
g = StarLayerGraph()
g.bind("ex", EX)

# create a triple term
tt = TripleTerm(EX.bob, EX.knows, EX.carol)

# create a reifier (ex:claim) associated with the triple term and add it to the graph
g.add_reification(EX.claim, tt)
g.add((EX.claim, EX.source, EX.wikipedia))

print((EX.claim, RDF.reifies, tt) in g)
print(g.serialize(format="turtle12"))

True
@version "1.2" .
@prefix ex: <http://example.org/> .
@prefix rdf: <http://www.w3.org/1999/02/22-rdf-syntax-ns#> .

ex:claim ex:source ex:wikipedia ;
    rdf:reifies <<( ex:bob ex:knows ex:carol )>> .



In [3]:
g = StarLayerGraph()
g.bind("ex", EX)

# add reifiers to the graph
g.add((EX.claim, RDF.reifies, (EX.bob, EX.knows, EX.carol)))
g.add((EX.other, RDF.reifies, (EX.bob, EX.likes, EX.dana)))

g.add((EX.bob, EX.knows, EX.dana))

# rdflib triples() now accepts a triple term as the object when selecting triples
selectTriples = g.triples((None, None, (EX.bob, EX.knows, EX.carol)))

for s, p, o in selectTriples:
    print(g.qname(s), g.qname(p), o)
for t in g.triple_terms(subject=EX.bob):
    print(t)

# has_triple_term() tests whether the triple term is in the graph.
# (EX.bob, EX.knows, EX.dana) is asserted directly in the graph, but is not the
# object of any triple, so it returns False.
print(g.has_triple_term(EX.bob, EX.knows, EX.carol))
print(g.has_triple_term(EX.bob, EX.knows, EX.dana))

ex:claim rdf:reifies <<( ex:bob ex:knows ex:carol )>>
<<( ex:bob ex:knows ex:carol )>>
<<( ex:bob ex:likes ex:dana )>>
True
False


In [4]:
# rdf:reifies is the common approach to making a statement about a statement.
# RDF 1.2 allows triple terms in the object position of any triple.

# add a triple to the graph with a triple term as the object
g.add((EX.dana, EX.said, (EX.bob, EX.knows, EX.carol)))

# triples() accepts a triple term as object to select matching triples
selectTriples = g.triples((None, None, (EX.bob, EX.knows, EX.carol)))

# qname_term() is a starlayer function that adds qname transformation to triple terms too.
for s, p, o in selectTriples:
    print(g.qname_term(s), g.qname_term(p), g.qname_term(o))

ex:claim rdf:reifies <<( ex:bob ex:knows ex:carol )>>
ex:dana ex:said <<( ex:bob ex:knows ex:carol )>>


### Finding what's been said about a statement

`reifiers()`, `reifications()`, `reifier_annotations()`, `reified_triples()`, and `remove_reification()` navigate the reifier/triple-term/annotation relationships directly, without SPARQL queries. `remove_reification(reifier, triple_term=None)` can be scoped to one specific reifier↔triple link, leaving any other triple(s) the same reifier reifies — and all its annotations — untouched; omit `triple_term` for the original all-or-nothing behavior.

In [5]:
# continues using g from the previous cell

# adds assertions to the reification ex:claim
g.add((EX.claim, EX.source, EX.wikipedia))

tt1 = (EX.bob, EX.knows, EX.carol)
tt2 = (EX.bob, EX.likes, EX.dana)

# reifiers(): returns the reifier node(s) that reify a given triple term.
print([g.qname(r) for r in g.reifiers(TT=tt1)])

# reifications(): returns a list of triple terms that have at least one reifier.
for tt in g.reifications():
    print(tt)

# reifier_annotations(): returns a reifier's annotation triples (excludes rdf:reifies itself)
for reifier, pred, val in g.reifier_annotations(tt1):
    print(g.qname(reifier), g.qname(pred), g.qname(val))

# reified_triples(): returns the triple term(s) a specific reifier reifies
for tt in g.reified_triples(EX.claim):
    print(tt)

# give ex:claim a second rdf:reifies link, so scoped vs. wildcard removal are distinguishable
g.add((EX.claim, RDF.reifies, tt2))

# remove_reification(reifier, triple_term): remove the reification link between a reifier
# and the specified triple term only.
g.remove_reification(EX.claim, tt1)
print((EX.claim, RDF.reifies, tt1) in g)         # False - only this link removed
print((EX.claim, RDF.reifies, tt2) in g)         # True  - untouched
print((EX.claim, EX.source, EX.wikipedia) in g)  # True  - untouched

# remove_reification(reifier): removes all rdf:reifies links from the reifier
g.remove_reification(EX.claim)
print((EX.claim, RDF.reifies, tt2) in g)
print((EX.claim, EX.source, EX.wikipedia) in g)

['ex:claim']
<<( ex:bob ex:knows ex:carol )>>
<<( ex:bob ex:likes ex:dana )>>
ex:claim ex:source ex:wikipedia
<<( ex:bob ex:knows ex:carol )>>
False
True
True
False
True


### Direction-tagged string literals

`DirLangString` sets the base direction of a language-tagged string literal.

In [6]:
g = StarLayerGraph()
g.bind("ex", EX)

# literals can now include language direction
g.add((EX.title, EX.value, DirLangString("مرحبا", "ar", "rtl")))
g.add((EX.title, EX.value, Literal("hello", "en")))
g.add((EX.title, EX.value, DirLangString("hello", "en", "ltr")))

print(g.serialize(format="turtle12"))

@version "1.2" .
@prefix ex: <http://example.org/> .

ex:title ex:value "hello"@en, "مرحبا"@ar--rtl, "hello"@en--ltr .



Adding directly to the dataset (`ds.add(...)`, no graph specified) writes to the dataset's own default graph, separate from any named graph. Serializing the whole dataset with a dataset format (`trig12`) shows every named graph, each in its own `GRAPH` block.

In [12]:
print(ds.serialize(format="trig12"))

@prefix ex: <http://example.org/> .

GRAPH <http://example.org/graph1> {
    ex:bob ex:knows ex:carol .

    ex:claim ex:source ex:wikipedia .
}

GRAPH <http://example.org/graph2> {
    ex:bob ex:likes ex:dana .
}



## 4. RDFC-1.0 canonicalization, hashing, and `isomorphic()`

Two graphs can represent the exact same information while disagreeing on blank node labels — labels are arbitrary identifiers, not part of the data. `isomorphic()` (`starlayergraph.compare`) checks graph equivalence correctly under that rule, RDF-1.2 triple terms included. `rdfc10_hash()` and `to_canonical_nquads()` (`starlayergraph.rdfc`) implement the [RDFC-1.0](https://www.w3.org/TR/rdf-canon/) canonicalization algorithm: a deterministic, blank-node-label-independent hash/serialization of a graph, so two isomorphic graphs always hash identically regardless of how their blank nodes happen to be labeled.

In [13]:
g1 = StarLayerGraph()
g1.bind("ex", EX)
b1 = BNode()
g1.add((EX.alice, EX.knows, b1))
g1.add((b1, EX.name, Literal("someone")))

# same shape, deliberately different (fresh, unrelated) blank node label
g2 = StarLayerGraph()
g2.bind("ex", EX)
b2 = BNode()
g2.add((EX.alice, EX.knows, b2))
g2.add((b2, EX.name, Literal("someone")))

print("isomorphic despite different blank node labels:", isomorphic(g1, g2))
print("hash g1:", rdfc10_hash(g1))
print("hash g2:", rdfc10_hash(g2))
print("hashes equal:", rdfc10_hash(g1) == rdfc10_hash(g2))

g3 = StarLayerGraph()
g3.bind("ex", EX)
g3.add((EX.alice, EX.knows, EX.bob))

print()
print("isomorphic to a graph with genuinely different data:", isomorphic(g1, g3))

isomorphic despite different blank node labels: True
hash g1: 236939de1885b1e84eef000f98a0803aeacd2611a57fe3cdac66ae4b9cc3cc1c
hash g2: 236939de1885b1e84eef000f98a0803aeacd2611a57fe3cdac66ae4b9cc3cc1c
hashes equal: True

isomorphic to a graph with genuinely different data: False


In [14]:
# to_canonical_nquads() is the canonical serialization the hash is derived from -
# blank nodes get deterministic c14n labels instead of their original arbitrary ones.
print(to_canonical_nquads(g1))

<http://example.org/alice> <http://example.org/knows> _:c14n0 .
_:c14n0 <http://example.org/name> "someone" .



This combination — extract exactly the part of a graph that matters, hash it canonically, and later re-verify the hash still matches — is the basis of the "commit a shape-scoped subgraph, detect if it later changes" workflow in the [subgraph extraction guide](04f-shacl-subgraph-extraction.ipynb)'s final section. That guide builds `commit_subgraph()`/`verify_commitment()` helpers on top of exactly the two functions shown here (`rdfc10_hash()` for the hash, `isomorphic()` for comparing extractions before/after a change).

## Further work

- **`StarLayerDataset` and blank nodes on a remote backend.** The examples above use the default in-memory store. Writing blank nodes to a dataset backed by a remote SPARQL store (Oxigraph, Fuseki) goes through the same skolemization handling documented in [`starlayergraph.md`](../../packages/graph/docs/starlayergraph.md)'s "Blank nodes against a remote store" section — see [the backend guide](06-backend-graph-databases.ipynb) for the full picture.
- **JSON-LD has no published RDF 1.2 spec yet.** `jsonld12` round-trips correctly against starlayer's own writer/parser, but shouldn't be treated as a spec-conformant interchange format outside this project.